# Preparación del dataset para fine-tuning QLoRA

Este notebook convierte el dataset de clasificación al formato instrucción–respuesta
necesario para el fine-tuning con QLoRA (notebook 12).

Pasos:
1. Cargar `data/dataset_riesgo.csv` (500 muestras)
2. Formatear cada muestra como par instrucción–respuesta
3. Split train (80%) / test (20%) estratificado
4. Guardar en `data/finetune/train.jsonl` y `data/finetune/test.jsonl`
5. Registro en MLflow

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

# Localizar src/classifier/ de forma robusta:
# - desde la raiz del proyecto (VS Code / Jupyter server)
# - desde la carpeta del notebook (Jupyter clasico)
_cwd = os.getcwd()
_candidates = [
    os.path.join(_cwd, "src", "classifier"),
    os.path.abspath(".."),
    os.path.abspath("."),
]
for _p in _candidates:
    if os.path.isfile(os.path.join(_p, "functions.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break

import functions
functions.MLFLOW_EXPERIMENT = "clasificador_riesgo_ia_artificial"

## 1. Carga del dataset

In [14]:
import pandas as pd

df = pd.read_csv("datasets/dataset_riesgo.csv")

print(f"Total de muestras: {len(df)}")
print(f"Columnas: {df.columns.tolist()}")
print()
print("Distribución por clase:")
print(df["etiqueta"].value_counts())

Total de muestras: 300
Columnas: ['id', 'descripcion', 'sector', 'tipo_datos', 'etiqueta', 'notas']

Distribución por clase:
etiqueta
alto_riesgo        90
inaceptable        77
riesgo_limitado    67
riesgo_minimo      66
Name: count, dtype: int64


## 2. Formato instrucción–respuesta

Cada muestra se convierte al formato:
```
### Instrucción:
Clasifica el siguiente sistema de IA según el Reglamento de Inteligencia Artificial de la UE
en una de estas categorías: inaceptable, alto_riesgo, riesgo_limitado, riesgo_minimo.

### Descripción:
{descripcion}

### Clasificación:
{etiqueta}
```

In [15]:
INSTRUCCION = (
    "Clasifica el siguiente sistema de IA según el Reglamento de Inteligencia Artificial "
    "de la UE en una de estas categorías: inaceptable, alto_riesgo, riesgo_limitado, riesgo_minimo."
)

def formatear_muestra(row):
    return (
        f"### Instrucción:\n{INSTRUCCION}\n\n"
        f"### Descripción:\n{row['descripcion']}\n\n"
        f"### Clasificación:\n{row['etiqueta']}"
    )

df["text"] = df.apply(formatear_muestra, axis=1)

print("Ejemplo de muestra formateada:")
print("-" * 60)
print(df["text"].iloc[0])

Ejemplo de muestra formateada:
------------------------------------------------------------
### Instrucción:
Clasifica el siguiente sistema de IA según el Reglamento de Inteligencia Artificial de la UE en una de estas categorías: inaceptable, alto_riesgo, riesgo_limitado, riesgo_minimo.

### Descripción:
Sistema de puntuación social que clasifica a ciudadanos según su historial de pagos, multas y comportamiento en redes para restringir su acceso a transporte público y servicios estatales

### Clasificación:
inaceptable


## 3. Split train / test estratificado

In [16]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["etiqueta"],
)

print(f"Train: {len(train_df)} muestras")
print(f"Test:  {len(test_df)} muestras")
print()
print("Distribución train:")
print(train_df["etiqueta"].value_counts())
print()
print("Distribución test:")
print(test_df["etiqueta"].value_counts())

Train: 240 muestras
Test:  60 muestras

Distribución train:
etiqueta
alto_riesgo        72
inaceptable        61
riesgo_limitado    54
riesgo_minimo      53
Name: count, dtype: int64

Distribución test:
etiqueta
alto_riesgo        18
inaceptable        16
riesgo_minimo      13
riesgo_limitado    13
Name: count, dtype: int64


## 4. Guardar en formato JSONL

In [17]:
import json
import os

output_dir = "data/finetune"
os.makedirs(output_dir, exist_ok=True)

def guardar_jsonl(df, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            record = {
                "text":     row["text"],
                "etiqueta": row["etiqueta"],
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

path_train = os.path.join(output_dir, "train.jsonl")
path_test  = os.path.join(output_dir, "test.jsonl")

guardar_jsonl(train_df, path_train)
guardar_jsonl(test_df,  path_test)

print(f"✓ Train guardado en: {path_train} ({len(train_df)} muestras)")
print(f"✓ Test  guardado en: {path_test}  ({len(test_df)} muestras)")

✓ Train guardado en: data/finetune\train.jsonl (240 muestras)
✓ Test  guardado en: data/finetune\test.jsonl  (60 muestras)


## 5. Verificación — longitud de secuencias

Comprobamos que las muestras caben en el contexto del modelo base (`max_length=512`).

In [18]:
# Estimación aproximada de tokens (1 token ≈ 4 caracteres en español)
df["tokens_aprox"] = df["text"].str.len() / 4

print("Estadísticas de longitud aproximada en tokens:")
print(df["tokens_aprox"].describe().round(1))
print()
superan_512 = (df["tokens_aprox"] > 512).sum()
print(f"Muestras que superan 512 tokens (aprox.): {superan_512} de {len(df)}")
if superan_512 > 0:
    print("⚠ Estas muestras serán truncadas durante la tokenización.")

Estadísticas de longitud aproximada en tokens:
count    300.0
mean     108.4
std        4.5
min       98.2
25%      105.8
50%      108.4
75%      111.6
max      121.0
Name: tokens_aprox, dtype: float64

Muestras que superan 512 tokens (aprox.): 0 de 300


## 6. Registro en MLflow

In [22]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
import mlflow
from functions import configure_mlflow, MLFLOW_EXPERIMENT

try:
    configure_mlflow()
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    with mlflow.start_run(run_name="preparacion_dataset_finetune"):
        mlflow.log_params({
            "n_muestras":    len(df),
            "formato":       "instruccion",
            "train_split":   0.8,
            "n_clases":      df["etiqueta"].nunique(),
            "max_length":    512,
        })

        for clase, n in df["etiqueta"].value_counts().items():
            mlflow.log_metric(f"n_{clase}", n)

        mlflow.log_metric("n_train", len(train_df))
        mlflow.log_metric("n_test",  len(test_df))
        mlflow.log_metric("tokens_aprox_media", df["tokens_aprox"].mean())

        mlflow.log_artifact(path_train)
        mlflow.log_artifact(path_test)

        print("✓ Dataset de fine-tuning registrado en MLflow")
        print(f"  Run ID: {mlflow.active_run().info.run_id}")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")

OSError: [WinError 1114] Error en una rutina de inicialización de biblioteca de vínculos dinámicos (DLL). Error loading "c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.